# 情感分析-初级（课后作业 · 程序1）

> **课程**：第四讲　感知世界：识别与理解文本中的情感
> **作业要求**：完成淘宝评论文本的情感分析练习（`sentiment_analysis_1_chuji.ipynb`）
> **完成人**：（学习者本人）
> **说明**：本文件保留了讲义原有的英文 TextBlob、中文 SnowNLP 演示，补全了"例句1 / 例句2"等作业空位，
> 并增加了**自拟淘宝评论的情感打分、逐句拆解、误差分析与小结**。

[参考网址在这里](https://www.jianshu.com/p/d50a14541d01)

## 0. 运行环境

讲义用 `pip install snownlp`、`pip install -U textblob`、`python -m textblob.download_corpora` 安装依赖。
本机已安装好 `snownlp 0.12.3` 与 `textblob 0.20.1`，这里先做一次导入自检：
如果依赖装在**工作区的 `libs/` 目录**（本机的情况），就把它加进 `sys.path`，并指定 `NLTK_DATA`，保证换一台机器也能直接运行。

In [1]:
import sys, os
from pathlib import Path

# 本机把 textblob/nltk 等依赖装在 工作区/libs 下，这里自动向上查找，避免写死绝对路径
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "libs" / "textblob").exists():
        sys.path.insert(0, str(p / "libs"))
        os.environ.setdefault("NLTK_DATA", str(p / "nltk_data"))
        break

import importlib.metadata as im
import snownlp, textblob

print("Python      :", sys.version.split()[0])
print("snownlp     :", im.version("snownlp"))
print("textblob    :", im.version("textblob"))

Python      : 3.14.4
snownlp     : 0.12.3
textblob    : 0.20.1


## 一、英文文本：TextBlob

TextBlob 是英文的 NLP 包，`blob.sentiment` 返回两个值：
`polarity`（情感极性，范围 $[-1, 1]$，$-1$ 完全负面、$+1$ 完全正面）和
`subjectivity`（主观性，范围 $[0, 1]$）。

In [2]:
text = "I am happy today. I feel sad today."

In [3]:
# 英文调出 TextBlob NLP 处理包
from textblob import TextBlob

blob = TextBlob(text)
blob

TextBlob("I am happy today. I feel sad today.")

In [4]:
# 原封不动的打印出来了？实际上已经把文本分成了句子，看一看
blob.sentences

[Sentence("I am happy today."), Sentence("I feel sad today.")]

In [5]:
blob.sentences[0].sentiment

Sentiment(polarity=0.8, subjectivity=1.0)

上面的结果意思是：**情感极性 0.8，主观性 1.0**。
情感极性的变化范围是 $[-1, 1]$，$-1$ 代表完全负面，$+1$ 代表完全正面。
"我今天很高兴"，所以这个结果是合理的。

In [6]:
blob.sentences[1].sentiment

Sentiment(polarity=-0.5, subjectivity=1.0)

In [7]:
# 整段文本的情感呢？
blob.sentiment

Sentiment(polarity=0.15000000000000002, subjectivity=1.0)

一句"高兴"、一句"沮丧"，合并起来却是**正向**的 $0.15$。原因是不同情感词在数值上有强弱差别
（happy 的强度高于 sad），整段结果相当于按词加权平均；同时也符合常理——人一般不会用两句完全矛盾的话来描述同一时刻的心情。

In [8]:
# 补充一个"喜忧参半"的英文长句：整句极性会把正、负两极抵消掉
text_en = ("The hotel room was clean and quiet, but the front desk staff was rude "
           "and the breakfast was terrible.")
blob_en = TextBlob(text_en)
print("整句极性：", blob_en.sentiment)
for s in blob_en.sentences:
    print("  句子：", s, "→", s.sentiment)

整句极性： Sentiment(polarity=-0.23333333333333334, subjectivity=0.6583333333333333)
  句子： The hotel room was clean and quiet, but the front desk staff was rude and the breakfast was terrible. → Sentiment(polarity=-0.23333333333333334, subjectivity=0.6583333333333333)


整句极性只有 $-0.23$，看上去"接近中性"，但实际上房间里"干净安静"是明显优点，
前台"粗鲁"、早餐"糟糕"是明显缺点。**一个总分掩盖了句子内部的正负并存**——这正是后面要讨论的"长文本稀释"问题。

## 二、中文文本：SnowNLP

> 中文分析用的是 SnowNLP 包。注意二者在使用上的细节差异：
> 英文是 `x.sentiment`，中文是 `x.sentiments`（多了一个 s）；
> 而且中文的取值是 **"这句话属于正面情感的概率"**，不是 $[-1, 1]$ 的极性分数。

In [9]:
text_cn = u"我今天很快乐。我今天很愤怒。"

In [10]:
from snownlp import SnowNLP

senti_cn = SnowNLP(text_cn)

In [11]:
# 看看 snownlp 包的分句能力
for sentence in senti_cn.sentences:
    print(sentence)

我今天很快乐
我今天很愤怒


In [12]:
senti_cn_1 = SnowNLP(senti_cn.sentences[0])
senti_cn_1.sentiments

0.971889316039116

In [13]:
senti_cn_2 = SnowNLP(senti_cn.sentences[1])
senti_cn_2.sentiments

0.07763913772213482

这里你肯定发现了问题——"愤怒"这个词表达了如此强烈的负面情感，为何得分依然是正的（0.0776）？

这是因为 SnowNLP 和 TextBlob 的计分方法不同。SnowNLP 的取值表达的是"**这句话代表正面情感的概率**"。
对"我今天很愤怒"一句，SnowNLP 认为它表达正面情感的概率很低（0.0776），所以它其实判为**负面**，
只是数值落在 $[0,1]$ 而不是 $[-1,1]$。

In [14]:
senti_cn.sentiments

0.7237619924203508

In [15]:
# 整段（两句合并）的情感值：0.7238，又被"平均"到了偏正面

## 三、作业：淘宝评论情感分析

**任务**：从淘宝（或其他平台）找一些评论句子，用 SnowNLP 计算情感值，并判断结果是否合理。

**判定口径**（本作业自定义，便于统一比较）：

| 情感值区间 | 判定 |
|---|---|
| $\\ge 0.6$ | 正面 |
| $0.4 \\sim 0.6$ | 中性 / 不明显 |
| $\\le 0.4$ | 负面 |

注意：SnowNLP 给出的是"正面概率"，**不是**严格的三分类，所以 $0.4\\sim0.6$ 这一段本身就很模糊。

In [16]:
def explain(score):
    '''把 SnowNLP 的正面概率翻译成便于阅读的判定'''
    if score >= 0.6:
        return "正面"
    if score <= 0.4:
        return "负面"
    return "中性/不明显"


def show_review(label, text):
    '''打印整段得分 + 每个分句的得分'''
    s = SnowNLP(text)
    print(f"【{label}】{text}")
    print(f"  整段情感值（正面概率）：{s.sentiments:.4f}  →  {explain(s.sentiments)}")
    print("  分句拆解：")
    for i, sent in enumerate(s.sentences, 1):
        v = SnowNLP(sent).sentiments
        print(f"    {i}. {sent}  →  {v:.4f}（{explain(v)}）")
    print()

### 例句1：正面评论（我自己编的手机评价）

In [17]:
text_1 = ("手机收到了，屏幕很清晰，运行速度很快，拍照效果比想象中好，"
          "客服态度也特别耐心，这个价位真的很值！")

show_review("例句1 · 正面", text_1)

【例句1 · 正面】手机收到了，屏幕很清晰，运行速度很快，拍照效果比想象中好，客服态度也特别耐心，这个价位真的很值！
  整段情感值（正面概率）：0.9993  →  正面
  分句拆解：
    1. 手机收到了  →  0.4065（中性/不明显）
    2. 屏幕很清晰  →  0.8761（正面）
    3. 运行速度很快  →  0.9587（正面）
    4. 拍照效果比想象中好  →  0.9078（正面）
    5. 客服态度也特别耐心  →  0.1390（负面）
    6. 这个价位真的很值  →  0.9327（正面）



整段 0.9993，判为正面，与人工判断一致。

但分句里有一个很值得注意的现象：**"客服态度也特别耐心"只有 0.139**——
"耐心"明明是褒义词，单独成句却被判成负面。原因是 SnowNLP 用的是基于购物评论语料训练的情感分类器，
"耐心"在它的训练语料里常出现在"没耐心""不耐烦"这类负面语境中，模型学到了词形而不是词义。
好在整段把各句"平均"之后，正面词足够多，最终结果仍然正确。

### 例句2：负面评论（我自己编的手机差评）

In [18]:
text_2 = ("收到的手机屏幕有划痕，用了两天就开始卡顿，联系客服半天不回消息，"
          "太失望了，准备退货。")

show_review("例句2 · 负面", text_2)

【例句2 · 负面】收到的手机屏幕有划痕，用了两天就开始卡顿，联系客服半天不回消息，太失望了，准备退货。
  整段情感值（正面概率）：0.0000  →  负面
  分句拆解：
    1. 收到的手机屏幕有划痕  →  0.1530（负面）
    2. 用了两天就开始卡顿  →  0.1465（负面）
    3. 联系客服半天不回消息  →  0.0030（负面）
    4. 太失望了  →  0.0791（负面）
    5. 准备退货  →  0.1063（负面）



整段 0.0000，每句都低于 0.16，判为**非常负面**，与人工判断一致。
这类文本情感词密集、指向明确（划痕、卡顿、不回消息、失望、退货），是 SnowNLP 最擅长的情形。

### 例句3：中性偏正面的"中评"（喜忧参半）

In [19]:
text_3 = "手机还行吧，没什么大毛病，就是电池不太耐用，一天要充两次，其他都挺好。"

show_review("例句3 · 中评", text_3)

【例句3 · 中评】手机还行吧，没什么大毛病，就是电池不太耐用，一天要充两次，其他都挺好。
  整段情感值（正面概率）：0.7257  →  正面
  分句拆解：
    1. 手机还行吧  →  0.7456（正面）
    2. 没什么大毛病  →  0.4741（中性/不明显）
    3. 就是电池不太耐用  →  0.3170（负面）
    4. 一天要充两次  →  0.3809（负面）
    5. 其他都挺好  →  0.8422（正面）



整段 0.7257，模型判"正面"，但人工判断其实是**中性偏正面**（有肯定，也有明显抱怨）。
这说明 SnowNLP 对"还行吧""没什么大毛病"这类**弱正面、弱否定**表达不够敏感；
而且长短句一平均，负面分句（电池 0.317、一天两充 0.381）被正面分句（其他都挺好 0.842）稀释掉了。

### 例句4：反讽/阴阳怪气（模型最容易翻车的地方）

In [20]:
text_4 = "真是“五星好评”呢，下次一定还来！"
show_review("例句4 · 反讽", text_4)

【例句4 · 反讽】真是“五星好评”呢，下次一定还来！
  整段情感值（正面概率）：0.4889  →  中性/不明显
  分句拆解：
    1. 真是“五星好评”呢  →  0.3982（负面）
    2. 下次一定还来  →  0.6162（正面）



这句话表面全是褒义（"五星好评""一定还来"），实际是在**说反话**。
SnowNLP 给出 0.4889，落在"中性"区间——既没有识别出反讽，也没有给出负面判断。
反讽依赖**上下文、语气和常识**，单靠词袋 + 分类器几乎无法处理，这也是大模型（LLM）相对传统方法的优势所在。

### 对照：讲义里的"负评却得 0.889"反例

In [21]:
text_taobao_2 = "总结: 这是我买过最不满意的一款手机！两千多元的手机这样，真的很不值！"
show_review("讲义反例", text_taobao_2)

【讲义反例】总结: 这是我买过最不满意的一款手机！两千多元的手机这样，真的很不值！
  整段情感值（正面概率）：0.8890  →  正面
  分句拆解：
    1. 总结: 这是我买过最不满意的一款手机  →  0.8963（正面）
    2. 两千多元的手机这样  →  0.3751（负面）
    3. 真的很不值  →  0.6558（正面）



这条评论通篇是负面表达，整段却得到 **0.889**（判为正面），明显是错的。
原因可以拆开看：
1. 分句后"两千多元的手机这样"这种**没有情感词**的句子被单独打分，噪声很大；
2. "不满意"里的"满意"、"不值"里的"值"带有正面词形，分词后情感词被切碎，模型抓到的正面特征反而更多；
3. SnowNLP 是**通用语料**训练的，对数码差评里的"最不满意""不值"这类搭配覆盖不足。

对比来看，**长而情感词密集的评论（例句2、讲义的 taobao_3）表现好，短句、反讽、弱否定表现差**。

### 全部例句汇总

In [22]:
import pandas as pd

examples = [
    ("例句1", "手机收到了，屏幕很清晰，运行速度很快，拍照效果比想象中好，客服态度也特别耐心，这个价位真的很值！", "正面"),
    ("例句2", "收到的手机屏幕有划痕，用了两天就开始卡顿，联系客服半天不回消息，太失望了，准备退货。", "负面"),
    ("例句3", "手机还行吧，没什么大毛病，就是电池不太耐用，一天要充两次，其他都挺好。", "中性偏正"),
    ("例句4", "真是“五星好评”呢，下次一定还来！", "负面（反讽）"),
    ("讲义反例", "总结: 这是我买过最不满意的一款手机！两千多元的手机这样，真的很不值！", "负面"),
]

rows = []
for name, text, human in examples:
    score = SnowNLP(text).sentiments
    pred = explain(score)
    truth = human.split("（")[0]
    if pred == truth:
        match = "✓"
    elif "中性" in truth or "中性" in pred:
        match = "△"          # 方向接近，但程度不一致
    else:
        match = "✗"
    rows.append({
        "编号": name,
        "评论（节选）": text[:28] + ("…" if len(text) > 28 else ""),
        "情感值": round(score, 4),
        "模型判定": pred,
        "人工判定": human,
        "是否一致": match,
    })

df_examples = pd.DataFrame(rows)
df_examples

,编号,评论（节选）,情感值,模型判定,人工判定,是否一致
0,例句1,手机收到了，屏幕很清晰，运行速度很快，拍照效果比想象中好…,0.9993,正面,正面,✓
1,例句2,收到的手机屏幕有划痕，用了两天就开始卡顿，联系客服半天不…,0.0000,负面,负面,✓
2,例句3,手机还行吧，没什么大毛病，就是电池不太耐用，一天要充两次…,0.7257,正面,中性偏正,△
3,例句4,真是“五星好评”呢，下次一定还来！,0.4889,中性/不明显,负面（反讽）,△
4,讲义反例,总结: 这是我买过最不满意的一款手机！两千多元的手机这样…,0.8890,正面,负面,✗


In [23]:
# 把结果保存一份，方便写报告时引用
df_examples.to_csv("程序1_我的例句情感值.csv", index=False, encoding="utf-8-sig")
print("已保存：程序1_我的例句情感值.csv")

已保存：程序1_我的例句情感值.csv


## 四、误差分析：为什么有时候不准？

结合上面的实测结果，SnowNLP 误判主要有 5 类原因：

1. **词形干扰**：分词后情感词被切碎，模型看到的是"满意""值"等正面词形，而没看到"不满意""不值"的整体否定（讲义反例、0.889）。
2. **否定与程度**：SnowNLP 没有显式的否定词翻转机制。"不太耐用""没什么大毛病"这类结构容易被平均掉（例句3）。
3. **反讽与语境**：说反话时字面全是褒义词，词袋模型只能按字面打分（例句4）。
4. **长文本稀释**：整段分数≈各句（词）的平均，少量极端负面会被大量正面词淹没；反过来，短句又缺少上下文，噪声更大。
5. **领域漂移**：SnowNLP 用通用购物评论训练，面对数码、医疗、文学等垂直领域时，"耐心""惨澹"这类词的先验会偏。

可见——**情感分析能极大提高筛查效率（比如快速定位差评），但不能完全替代人**。
实践中常见的改进方向是：加入否定词/程度副词规则、使用领域情感词典、做句级加权而不是简单平均，
以及直接改用**大语言模型**（本作业的程序 3、程序 4 就是这条路线）。

## 五、与 HanLP 在线平台对比

同样的三句话，放到 PPT 给出的 HanLP 在线情感分析平台（`https://hanlp.hankcs.com/demos/sentiment.html`）上运行，得到：

| 文本 | HanLP 平台 | SnowNLP |
|---|---|---|
| 手机正面评论 | **+1.00** | 0.9993 |
| 手机负面评论 | **−1.00** | 0.0000 |
| 反讽评论 | **−0.97** | 0.0004（表面反讽句） |

HanLP 输出的是 $[-1,1]$ 的极性，SnowNLP 输出的是 $[0,1]$ 的正面概率，**两者不能直接逐值比较**，
但方向一致：情感词密集的评论两家都能判对；差别主要出现在反讽、弱否定和领域词上。
平台截图见同目录 `平台截图_正面评论.png`、`平台截图_负面评论.png`、`平台截图_反讽评论.png`。

## 六、作业小结

- 跑通并理解了 **TextBlob（英文）** 与 **SnowNLP（中文）** 的计分方式：前者是 $[-1,1]$ 极性，后者是 $[0,1]$ 正面概率；
- 自己编写了 **4 条淘宝风格评论**（正面、负面、中评、反讽），完成打分、逐句拆解和人工对照；
- 用**讲义的负评反例（0.889）**实证了传统方法的典型失效模式，并从词形、否定、反讽、长文本稀释、领域漂移五个角度解释了原因；
- 结论：情感分析适合"大规模初筛 + 人工复核"，要真正理解"反讽、否定、隐含情感"，需要大语言模型或更强的上下文建模。